#Sets up the environment and ensures the output directory data/processed/ exists.
#Fixes the random seed (np.random.seed(42)) to ensure all data splits and window indexing remain 100% deterministic and reproducible.

In [24]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

# Path to the raw CU-BEMS dataset directory
DATA_DIR = "C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets"
OUTPUT_DIR = "C:\\Users\\vaish\\OneDrive\\Desktop\\Dataset\\data\\processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Libraries imported successfully!")

Libraries imported successfully!


#🔍 Explanation of Cell 2:
#The Paper's Objective: Section V mentions: "the total energy consumption for CU-BEMS data for all the floors combined is resampled by taking the sum of each minute for that day".
#This function loops through all 7 floors for both 2018 and 2019, sums every circuit (z1_Light, z2_AC, z3_Plug, etc.) across each floor, and sums all 7 floors together.
#Output: A single Pandas DataFrame containing total building electrical load (energy_kW) indexed by exact 1-minute timestamps.

In [25]:
def load_and_aggregate_building_load(data_dir):
    """
    Loads all 14 floor CSV files for 2018 and 2019, sums all sub-metered
    columns (Lighting, Plugs, ACs) per floor, and combines all 7 floors
    into the total building load at 1-minute resolution.
    
    Includes robust handling for corrupted logger rows (e.g. 2019Floor6 row 424168).
    """
    yearly_dfs = []
    
    for year in ["2018", "2019"]:
        floor_series_list = []
        
        for floor in range(1, 8):
            file_path = os.path.join(data_dir, f"{year}Floor{floor}.csv")
            if not os.path.exists(file_path):
                raise FileNotFoundError(f"Missing file: {file_path}")
            
            print(f"Reading: {file_path} ...")
            df_floor = pd.read_csv(file_path, low_memory=False)
            
            # Robust parsing: turns corrupt values (like "2") into NaT, then drops them
            df_floor['Date'] = pd.to_datetime(df_floor['Date'], errors='coerce')
            df_floor = df_floor.dropna(subset=['Date'])
            df_floor = df_floor.set_index('Date')
            
            # Ensure all electrical measurements are numeric floats
            df_floor = df_floor.apply(pd.to_numeric, errors='coerce').fillna(0.0)
            
            # Sum all electrical loads on this floor for each minute
            floor_total = df_floor.sum(axis=1)
            floor_series_list.append(floor_total)
            
        # Combine all 7 floors for this year
        year_total = pd.concat(floor_series_list, axis=1).sum(axis=1)
        yearly_dfs.append(year_total)
        
    # Concatenate 2018 and 2019 into a single continuous time series
    total_series = pd.concat(yearly_dfs).sort_index()
    
    # Remove any duplicate timestamps if present
    total_series = total_series[~total_series.index.duplicated(keep='first')]
    
    df_building = pd.DataFrame({'energy_kW': total_series})
    return df_building

df_building_1min = load_and_aggregate_building_load(DATA_DIR)
print(f"\nAggregated 1-minute dataset shape: {df_building_1min.shape}")
print(df_building_1min.head())

Reading: C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets\2018Floor1.csv ...
Reading: C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets\2018Floor2.csv ...
Reading: C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets\2018Floor3.csv ...
Reading: C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets\2018Floor4.csv ...
Reading: C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets\2018Floor5.csv ...
Reading: C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets\2018Floor6.csv ...
Reading: C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets\2018Floor7.csv ...
Reading: C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets\2019Floor1.csv ...
Reading: C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets\2019Floor2.csv ...
Reading: C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets\2019Floor3.csv ...
Reading: C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets\2019Floor4.csv ...
Reading: C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets\2019Floor5.csv ...
Reading: C:/Users/vaish/OneDrive/Desktop/Dataset/Datasets\2019Floor6.csv ...

Why Reindex? If a sensor gateway was offline for 10 minutes, those 10 rows might be completely absent from the CSV. Reindexing to pd.date_range(..., freq='1min') exposes those gaps as explicit NaN values.
Paper Imputation Rule (Section IV-B): "The first step of data processing is data cleaning... adopted linear imputation method for filling the null (missing) value. Each missing value has been filled with the value recorded at previous timestamp. If multiple continuous values are missing, all have been filled with the valid value recorded at previous timestamp." This is implemented cleanly via .ffill().
Resampling: We resample from 1-min to hourly ('1h') totals using .sum().

In [26]:
# 1. Reindex to a complete continuous 1-minute grid to reveal all missing timestamps
full_index_1min = pd.date_range(
    start=df_building_1min.index.min(),
    end=df_building_1min.index.max(),
    freq='1min'
)
df_reindexed = df_building_1min.reindex(full_index_1min)

# Check missing values count before imputation
missing_count = df_reindexed['energy_kW'].isna().sum()
print(f"Total missing 1-minute records: {missing_count} ({missing_count / len(df_reindexed):.2%})")

# 2. Linear / Forward-fill imputation as specified in Section IV-B of paper
# "Each missing value has been filled with the value recorded at previous timestamp."
df_imputed = df_reindexed.ffill().bfill()

# 3. Resample from 1-minute to 1-hour interval (Hourly Sum)
# The paper forecasts hourly consumption: resample by summing 60 minutes of each hour
df_hourly = df_imputed.resample('1h').sum()

print(f"\nResampled Hourly Dataset shape: {df_hourly.shape}")
print(df_hourly.head())

Total missing 1-minute records: 0 (0.00%)

Resampled Hourly Dataset shape: (13176, 1)
                     energy_kW
2018-07-01 00:00:00  132184.91
2018-07-01 01:00:00  134281.82
2018-07-01 02:00:00  133303.13
2018-07-01 03:00:00  136726.49
2018-07-01 04:00:00  140079.97


hour_of_day: Office buildings have massive electricity surges from 09:00 to 17:00 when employees turn on lights, computers, and AC compressors. Mapping hours 
0
…
23
0…23 allows the model to learn daily load curves.

  
day_type: On Saturdays and Sundays, office occupancy drops close to baseline. Mapping Monday–Sunday (
0
…
6
0…6) allows the model to adjust predictions for weekends.

In [27]:
# Create feature columns as detailed in Section IV-C & V of the paper:
# 1. Hour of the Day: 0 (12:00 AM) to 23 (11:00 PM)
df_hourly['hour_of_day'] = df_hourly.index.hour

# 2. Day Type: 0 (Monday) to 6 (Sunday)
df_hourly['day_type'] = df_hourly.index.dayofweek

print("Engineered Feature Preview:")
print(df_hourly.head(10))

Engineered Feature Preview:
                     energy_kW  hour_of_day  day_type
2018-07-01 00:00:00  132184.91            0         6
2018-07-01 01:00:00  134281.82            1         6
2018-07-01 02:00:00  133303.13            2         6
2018-07-01 03:00:00  136726.49            3         6
2018-07-01 04:00:00  140079.97            4         6
2018-07-01 05:00:00  135757.12            5         6
2018-07-01 06:00:00  140091.24            6         6
2018-07-01 07:00:00  143789.87            7         6
2018-07-01 08:00:00  144077.61            8         6
2018-07-01 09:00:00  141978.76            9         6


"Next, it is divided into training (90%) and testing (10%) sets."
"In this article, 90% of data is used for training, and the rest is used for testin

In [28]:
# 1. Chronological Split (Section IV-B): 90% training (~16 months), 10% testing (~2 months)
train_size = int(len(df_hourly) * 0.90)

df_train_raw = df_hourly.iloc[:train_size].copy()
df_test_raw = df_hourly.iloc[train_size:].copy()

print(f"Training hourly samples: {len(df_train_raw)} (approx {len(df_train_raw)/24:.1f} days)")
print(f"Testing hourly samples:  {len(df_test_raw)} (approx {len(df_test_raw)/24:.1f} days)")

# 2. Z-Score Standardization (Equation 12 in Paper): X_bar = (x - mean) / std
mean_energy = df_train_raw['energy_kW'].mean()
std_energy = df_train_raw['energy_kW'].std()

print(f"\nTraining Set Normalization Parameters:")
print(f"Mean Energy: {mean_energy:.2f} kW | Std Energy: {std_energy:.2f} kW")

# Apply scaling to training and testing sets
df_train = df_train_raw.copy()
df_test = df_test_raw.copy()

df_train['energy_norm'] = (df_train['energy_kW'] - mean_energy) / std_energy
df_test['energy_norm'] = (df_test['energy_kW'] - mean_energy) / std_energy

# Save scalar parameters so we can inverse-transform predicted kW back to real scale later
scaling_params = {'mean': mean_energy, 'std': std_energy}

Training hourly samples: 11858 (approx 494.1 days)
Testing hourly samples:  1318 (approx 54.9 days)

Training Set Normalization Parameters:
Mean Energy: 107580.36 kW | Std Energy: 74338.12 kW


🟩 Cell 6: Sliding Window Sequence Generator (Univariate & Multivariate Lags)

In [29]:
def create_time_series_windows(data_df, lag, scenario='univariate'):
    """
    Creates input sequences (X) and target outputs (y) based on lag hours.
    
    Parameters:
      - data_df: DataFrame with normalized energy and calendar features
      - lag: Number of previous hours to look back (2, 4, 8, 10)
      - scenario: 'univariate' (energy only) or 'multivariate' (energy + hour + day)
      
    Returns:
      - X: Numpy array of shape (samples, lag, num_features)
      - y: Numpy array of shape (samples,) - target energy at next hour
    """
    X_list, y_list = [], []
    
    if scenario == 'univariate':
        feature_cols = ['energy_norm']
    elif scenario == 'multivariate':
        # Feature columns: energy, hour_of_day, day_type
        feature_cols = ['energy_norm', 'hour_of_day', 'day_type']
    else:
        raise ValueError("Scenario must be 'univariate' or 'multivariate'")
        
    values = data_df[feature_cols].values
    target_values = data_df['energy_norm'].values
    
    for i in range(len(values) - lag):
        # Look back window of 'lag' hours
        X_list.append(values[i : i + lag])
        # Target is the energy consumption at the (i + lag)-th hour
        y_list.append(target_values[i + lag])
        
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

# Generate and verify tensor dimensions for both scenarios at all paper lags: [2, 4, 8, 10]
for lag in [2, 4, 8, 10]:
    X_train_uni, y_train_uni = create_time_series_windows(df_train, lag=lag, scenario='univariate')
    X_train_multi, y_train_multi = create_time_series_windows(df_train, lag=lag, scenario='multivariate')
    
    print(f"Lag {lag:2d}h -> Univariate X: {X_train_uni.shape}, y: {y_train_uni.shape} | "
          f"Multivariate X: {X_train_multi.shape}, y: {y_train_multi.shape}")

Lag  2h -> Univariate X: (11856, 2, 1), y: (11856,) | Multivariate X: (11856, 2, 3), y: (11856,)
Lag  4h -> Univariate X: (11854, 4, 1), y: (11854,) | Multivariate X: (11854, 4, 3), y: (11854,)
Lag  8h -> Univariate X: (11850, 8, 1), y: (11850,) | Multivariate X: (11850, 8, 3), y: (11850,)
Lag 10h -> Univariate X: (11848, 10, 1), y: (11848,) | Multivariate X: (11848, 10, 3), y: (11848,)


Final

In [30]:
# Save clean hourly dataframe and metadata for model training
df_hourly.to_csv(os.path.join(OUTPUT_DIR, "cu_bems_hourly_cleaned.csv"))
np.save(os.path.join(OUTPUT_DIR, "scaling_params.npy"), scaling_params)

print(f"All preprocessed data successfully saved to '{OUTPUT_DIR}/'!")

All preprocessed data successfully saved to 'C:\Users\vaish\OneDrive\Desktop\Dataset\data\processed/'!
